# gramlab · run 21 — more and broader data for the role linker

The one function that measurably works is the role linker (who did what to whom, 0.887 F1 on GUM) and the ordered key
built on it. This run widens its training data beyond AMALGUM's Wikipedia-family text and measures what that buys:

- **gold**: UD English treebanks with gold relations in other registers — ParTUT (legal, talks), LinES (literature, manuals,
  Europarl), Atis (flight dialogue) for training; GENTLE (dictionary, esports, legal, medical, poetry, proofs, syllabi,
  threat letters), PUD (news) and Pronouns as held-out out-of-domain tests. EWT is not used (project decision).
- **silver**: non-Wikipedia text parsed by Stanza (UD-native): Brown (15 genres), Gutenberg books, Reuters newswire,
  web forums, NPS chat — capped at ~40k sentences.

Arms: **A** AMALGUM only (run-9 recipe) · **B** + gold · **C** + silver · **D** + gold + silver. Thresholds chosen on GUM dev.
Scored on GUM test (gold tags and live tags) and on every extra treebank's test split. Then the reversal bench (run 17/20
construction) is re-run with arm D's linker + front end to see the downstream effect. CPU only; ~60–90 min.


In [ ]:
import os, sys, glob, subprocess, pathlib, json
WORK = pathlib.Path('/kaggle/working'); os.chdir(WORK)
def sh(c):
    print('$', c); r = subprocess.run(c, shell=True, text=True, capture_output=True)
    if r.returncode: print(r.stdout[-1500:], r.stderr[-1500:])
    return r.returncode == 0
sh('git clone --depth 1 https://github.com/furevych/gramlab.git gramlab')
sys.path.insert(0, str(WORK / 'gramlab'))
GUM = WORK / 'data' / 'gum'; GUM.mkdir(parents=True, exist_ok=True)
for split in ('train', 'dev', 'test'):
    sh(f'curl -sSL -o {GUM}/en_gum-ud-{split}.conllu https://raw.githubusercontent.com/UniversalDependencies/UD_English-GUM/master/en_gum-ud-{split}.conllu')
AMALGUM = 'data/amalgum/amalgum/*/dep/*.conllu'
if not glob.glob(AMALGUM):
    sh('GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/gucorpling/amalgum.git data/amalgum')
print('GUM', sorted(p.name for p in GUM.glob('*.conllu')), '| AMALGUM files', len(glob.glob(AMALGUM)))
# layer-0 preset: WordNet 3.0 files, read by gramlab.core.wordnet (no nltk needed)
if not pathlib.Path('data/wordnet/index.verb').exists():
    sh('curl -sSL -o data/wordnet.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/wordnet.zip')
    import zipfile; zipfile.ZipFile('data/wordnet.zip').extractall('data')      # -> data/wordnet/
os.environ['WORDNET_DIR'] = str(WORK / 'data' / 'wordnet')
from gramlab.core.wordnet import WordNet; print('WordNet', WordNet().stats())
if not pathlib.Path('data/semcor/brown2').exists():
    sh('curl -sSL -o data/semcor.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/semcor.zip')
    zipfile.ZipFile('data/semcor.zip').extractall('data')                        # -> data/semcor/
SEMCOR = str(WORK / 'data' / 'semcor')
# affect lexicon: attach a Kaggle dataset containing NRC-VAD-Lexicon.txt (word, valence, arousal, dominance) to enable the affect row
VAD = next(iter(glob.glob('/kaggle/input/**/NRC-VAD-Lexicon*.txt', recursive=True) + glob.glob('/kaggle/input/**/*vad*.txt', recursive=True)), None)
print('VAD lexicon:', VAD)


In [ ]:
# extra gold treebanks
UD = WORK / 'data' / 'ud'; UD.mkdir(parents=True, exist_ok=True)
EXTRA = {}
for tb in ('ParTUT', 'LinES', 'Atis', 'GENTLE', 'PUD', 'Pronouns'):
    d = UD / f'UD_English-{tb}'
    if not d.exists(): sh(f'git clone -q --depth 1 https://github.com/UniversalDependencies/UD_English-{tb}.git {d}')
    EXTRA[tb.lower()] = str(d)
print({k: sorted(p.name for p in pathlib.Path(v).glob('*.conllu')) for k, v in EXTRA.items()})

# non-Wikipedia raw text from nltk_data (GitHub), parsed to UD by Stanza
NL = WORK / 'data' / 'nltk'; NL.mkdir(exist_ok=True)
import zipfile
for name in ('brown', 'gutenberg', 'reuters', 'webtext', 'nps_chat'):
    if not (NL / name).exists():
        if sh(f'curl -sSL -o {NL}/{name}.zip https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/corpora/{name}.zip'):
            try: zipfile.ZipFile(NL / f'{name}.zip').extractall(NL)
            except Exception as e: print('unzip failed', name, e)
from gramlab.core import more_data
texts = more_data.nltk_texts(NL)
print({k: len(v) for k, v in texts.items()})

In [ ]:
# Stanza silver parses (UD tags + relations), capped per source so no genre dominates
SILVER = []
try:
    sh('pip -q install stanza 2>/dev/null || true')
    import stanza; stanza.download('en', verbose=False)
    import random; rng = random.Random(0)
    caps = {'brown': 14000, 'gutenberg': 10000, 'reuters': 8000, 'webtext': 4000, 'nps_chat': 4000}
    for name, tx in texts.items():
        tx = list(tx); rng.shuffle(tx)
        got = more_data.stanza_silver(tx, max_sents=caps.get(name, 5000), doc=name)
        print(name, len(got), more_data.role_stats(got)); SILVER += got
except Exception as e:
    print('silver skipped:', repr(e))
print('silver total', len(SILVER))

In [ ]:
board = more_data.run(str(GUM), AMALGUM, extra=EXTRA, silver_sents=SILVER, out_dir=str(WORK / 'results'))
import json; print(json.dumps({a: v['scores'] for a, v in board['arms'].items()}, indent=1))

In [ ]:
# downstream: the reversal bench with arm D's linker + front end (BM25 · key · soft map untrained)  (~5 min)
from gramlab.core import softmap
RES = WORK / 'results'
ret = softmap.run(AMALGUM, SEMCOR, str(RES / 'frontend.pkl'), str(RES / 'linker.pkl'), out_dir=str(RES), n_queries=2000,
                  variants=('roles/synsets/untrained',), key_bonus=1.0)
print(json.dumps({k: v for k, v in ret['systems'].items()}, indent=1)); print(ret['live_queries'])

## Reading it
- `gum_test` F1 across arms A→D: does broader data help or hurt on the fixed benchmark (run 9: 0.887)?
- `*_test` rows: out-of-domain generalisation — GENTLE and PUD were never trained on in any arm.
- `f1_live` and `upos_live`: the front end's tagger is retrained on the same mix; live F1 is what the key gets on raw text.
- bench: `roles_right` of 698 and `hybrid bm25+key` recall@1 (run 20 with the AMALGUM-only models: 637/698 and 0.891).
Record as run 21 in RESULTS.md.